In [41]:
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
import statsmodels.api as sm

# Daten einlesen

In [42]:
df = pd.read_csv("../data/processed/swissvotes_processed.csv")

# Unterschliedliche Positionen zwiscnen BR und BV

In [43]:
# Nur Abstimmungen wo BR und Parlament unterschiedlicher Meinung sind
dissens = df[df['br-pos'] != df['bv-pos']].dropna(subset=['br-pos', 'bv-pos'])
print(f"Anzahl Fälle wo BR-Pos != BV-Pos: {len(dissens)}")

Anzahl Fälle wo BR-Pos != BV-Pos: 12


In [44]:

dissens = df[df['br-pos'] != df['bv-pos']].dropna(subset=['br-pos', 'bv-pos'])
print(dissens.groupby('hauptgruppe')['titel_kurz_d'].count().sort_values(ascending=False))

hauptgruppe
Staatsordnung                       4
Wirtschaft                          3
Sozial- und Gesellschaftspolitik    2
Umwelt & Lebensraum                 2
Landwirtschaft                      1
Name: titel_kurz_d, dtype: int64


# Modell
Wie stark sollen Ja-Anteil und Stimmbeteiligung in den Zustimmungsfaktor einfliessen?


## Ansatz: Logistische Regression (VERWORFEN)
Zum Bestimmen der Gewichte, fitten wir eine logistische Regression.


In [45]:
temp = df[['volkja-proz', 'beteiligung', 'rechtsform_name']].dropna()
temp = pd.get_dummies(temp, columns=['rechtsform_name'], drop_first=True)
temp = temp.astype(float)

X = sm.add_constant(temp.drop(columns=['volkja-proz']))
y = temp['volkja-proz']

result = sm.OLS(y, X).fit()
print(result.summary())

                            OLS Regression Results                            
Dep. Variable:            volkja-proz   R-squared:                       0.425
Model:                            OLS   Adj. R-squared:                  0.421
Method:                 Least Squares   F-statistic:                     99.90
Date:                Wed, 08 Apr 2026   Prob (F-statistic):           8.58e-79
Time:                        19:31:22   Log-Likelihood:                -2737.7
No. Observations:                 681   AIC:                             5487.
Df Residuals:                     675   BIC:                             5515.
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                                                 coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------------------

Die Regression zeigt, dass der Ja-Anteil kaum durch die Mobilisierung beeinflusst wird. Es gibt keine datengetriebenes Gewichtung. Deshalb müssen wir uns ein anderes Konzept überlegen.

# Kongruenz Rechnung anhand eines ungewichteten Prozentsatzes

In [46]:
def kongruenz_prozent(df, pos_col, min_n=10):
    """
    Berechnet, wie viel Prozent der Stimmberechtigten
    im Schnitt der Empfehlung eines Akteurs gefolgt sind.

    Parameter:
    - df: DataFrame mit Abstimmungsdaten
    - pos_col: Spaltenname der Position (z.B. 'p-svp')
    - min_n: Mindestanzahl Abstimmungen für gültiges Ergebnis

    Rückgabe:
    - score: Anteil Stimmberechtigte, die im Schnitt folgten (0 bis 1)
    - n: Anzahl ausgewertete Abstimmungen
    """

    # === SCHRITT 1: Daten vorbereiten ===
    # Nur die drei relevanten Spalten behalten
    temp = df[[pos_col, 'volkja-proz', 'beteiligung']].dropna()

    # Sicherstellen, dass die Position eine Zahl ist
    temp[pos_col] = pd.to_numeric(temp[pos_col], errors='coerce')

    # Nur Abstimmungen behalten, wo eine klare Empfehlung vorliegt
    # 1.0 = Ja-Empfehlung, 2.0 = Nein-Empfehlung
    temp = temp[temp[pos_col].isin([1.0, 2.0])]

    # === SCHRITT 2: Genug Daten vorhanden? ===
    if len(temp) < min_n:
        return np.nan, len(temp)

    # === SCHRITT 3: Werte in Anteile umrechnen ===
    # Beispiel: 65% Ja-Stimmen wird zu 0.65
    ja_anteil = temp['volkja-proz'] / 100

    # Beispiel: 45% Beteiligung wird zu 0.45
    beteiligung = temp['beteiligung'] / 100

    # === SCHRITT 4: Wie viele haben gleich gestimmt wie empfohlen? ===
    # Wenn Partei Ja empfiehlt: der Ja-Anteil zählt
    # Wenn Partei Nein empfiehlt: der Nein-Anteil zählt (= 1 minus Ja-Anteil)
    #
    # Beispiel Ja-Empfehlung:  65% stimmten Ja → zustimmung = 0.65
    # Beispiel Nein-Empfehlung: 65% stimmten Ja → zustimmung = 0.35
    ist_ja_empfehlung = (temp[pos_col] == 1.0)
    zustimmung = np.where(ist_ja_empfehlung, ja_anteil, 1 - ja_anteil)

    # === SCHRITT 5: Auf alle Stimmberechtigten umrechnen ===
    # zustimmung bezieht sich nur auf die Stimmenden
    # Wir wollen aber den Anteil ALLER Stimmberechtigten
    #
    # Beispiel: 65% der Stimmenden sagten Ja, Beteiligung 45%
    # → 0.65 * 0.45 = 0.2925 → 29.25% aller Stimmberechtigten folgten
    gefolgt = zustimmung * beteiligung

    # === SCHRITT 6: Durchschnitt über alle Abstimmungen ===
    score = gefolgt.mean()

    return score, len(temp)

In [47]:
score, n = kongruenz_prozent(df, 'p-svp')
print(f"SVP: {score:.2%} der Stimmberechtigten folgten im Schnitt (n={n})")
score, n = kongruenz_prozent(df, 'p-mitte')
print(f"Mitte: {score:.2%} der Stimmberechtigten folgten im Schnitt (n={n})")
score, n = kongruenz_prozent(df, 'br-pos')
print(f"Bundesrat: {score:.2%} der Stimmberechtigten folgten im Schnitt (n={n})")
score, n = kongruenz_prozent(df, 'p-vcs')
print(f"VCS: {score:.2%} der Stimmberechtigten folgten im Schnitt (n={n})")

SVP: 27.19% der Stimmberechtigten folgten im Schnitt (n=594)
Mitte: 28.20% der Stimmberechtigten folgten im Schnitt (n=43)
Bundesrat: 28.16% der Stimmberechtigten folgten im Schnitt (n=554)
VCS: 21.86% der Stimmberechtigten folgten im Schnitt (n=20)


## Kongruenz ohne Mobilisierung (Framing nicht mehr Stimmberechtigte sondern in Bezug auf

In [64]:
df_with_positions = df.copy()
neue_spalten = {}

for col in parteien:
    scores = []

    for i, row in df_with_positions.iterrows():
        position = row[col]
        ja_proz = row['volkja-proz']

        # Keine klare Empfehlung oder keine Daten
        if position not in [1.0, 2.0] or pd.isna(ja_proz):
            scores.append(np.nan)

        # Partei empfiehlt JA → je mehr Ja-Stimmen, desto positiver
        elif position == 1.0:
            scores.append((ja_proz - 50) / 100)

        # Partei empfiehlt NEIN → je mehr Nein-Stimmen, desto positiver
        elif position == 2.0:
            scores.append((50 - ja_proz) / 100)

    neue_spalten[f'zustimmung_{col}'] = scores

df_with_positions = pd.concat(
    [df_with_positions, pd.DataFrame(neue_spalten, index=df_with_positions.index)],
    axis=1
)
print(df_with_positions['zustimmung_br-pos'].dropna().unique())

[ 0.0097 -0.0057 -0.1168 -0.0421  0.0147 -0.0159 -0.1184  0.2069 -0.289
 -0.1491 -0.0894 -0.0211 -0.0604 -0.1145  0.1586  0.0286 -0.294   0.0806
 -0.1888  0.2068 -0.0884 -0.0446  0.0583 -0.3012 -0.0667  0.1791 -0.1979
  0.0914  0.1496  0.0959 -0.1917  0.2562  0.1258  0.0522 -0.1348  0.0246
  0.0436  0.0407 -0.1678 -0.0019 -0.053   0.0712 -0.2136  0.1643  0.3409
  0.1192 -0.0537  0.3704  0.3899 -0.3152  0.2323 -0.0763  0.08    0.1221
 -0.0986  0.0192  0.4732  0.1642  0.1285  0.1735 -0.103  -0.0011 -0.0511
 -0.0381  0.0416 -0.1765  0.0716  0.1868  0.4159  0.3478  0.3848  0.0346
  0.2108 -0.1237 -0.0573  0.0977  0.1757  0.1505  0.0292  0.0671  0.3081
  0.1881  0.3    -0.1384 -0.2515 -0.0074  0.0528 -0.0371  0.2299 -0.0568
  0.3755 -0.0392  0.0405  0.3103  0.0626  0.1799 -0.3451 -0.1354  0.3132
 -0.1695 -0.0595  0.1876 -0.0024  0.1308 -0.0754  0.2411  0.1501  0.2518
  0.0634 -0.0345  0.2056  0.167   0.1518 -0.1832  0.1861  0.1199  0.2663
  0.1731 -0.0175 -0.1552  0.0424  0.0401  0.0108  0.

In [49]:
df_with_positions.to_csv('../data/processed/df_with_positions.csv', index=False)